In [ ]:
from pathlib import Path
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA
DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"
DATA_TICKER = DATA_DIR / "processed" / "layoffs_with_tickers.csv"


# Load layoff data
layoffs = pd.read_csv(DATA_TICKER)

# Load financial features
features_df = pd.read_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"]
)

# Convert layoff dates to quarters
layoffs["date"] = pd.to_datetime(layoffs["date"])

layoffs["quarter"] = (
    layoffs["date"]
    .dt.to_period("Q")
    .astype(str)
)

# Create layoff labels
layoff_labels = (
    layoffs[["Company_name", "quarter"]]
    .drop_duplicates()
    .assign(layoff=1)
)

# Shift one quarter backwards
layoff_labels["quarter"] = (
    pd.PeriodIndex(
        layoff_labels["quarter"],
        freq="Q"
    ) - 1
).astype(str)

# Merge labels with financial features
dataset = features_df.merge(
    layoff_labels,
    left_on=["company", "quarter"],
    right_on=["Company_name", "quarter"],
    how="left"
)

# Fill non-layoffs
dataset["layoff"] = (
    dataset["layoff"]
    .fillna(0)
    .astype(int)
)

dataset.drop(
    columns=["Company_name"],
    inplace=True
)
print(dataset["layoff"].value_counts())
print(dataset["layoff"].value_counts(normalize=True))
dataset.to_csv(
    MERGED_DATA["LABELED_OUTPUT_CSV_PATH"],
    index=False
)

layoff
0    12002
1       45
Name: count, dtype: int64
layoff
0    0.996265
1    0.003735
Name: proportion, dtype: float64


KeyError: 'LABELED_OUTPUT_CSV_PATH'